# SmartAutoDJ — interactive demo / scratchpad

Drive the real pipeline (`smartautodj.pipeline.run`) end-to-end, **listen** to each
tier in the browser, look at the plots, and compare the objective metrics.

| Tier | What it does |
|------|--------------|
| 1 | baseline naive linear crossfade |
| 2 | tempo-match + downbeat-aligned transition with fade/EQ curves |
| 3 | tier 2 + an additive procedural "bridge" layer (riser / drum fill) |

Runs **either in Google Colab or locally** — section 0 sets up whichever you're on.
Then run top-to-bottom. Re-run any cell after editing the **knobs** in section 0.

> **Local:** use the `smartdj` conda env (`conda activate smartdj`; if the kernel isn't
> listed, `python -m ipykernel install --user --name smartdj`).
> **Colab:** just run the setup cell below — it clones the repo and installs deps.

## 0. Environment setup (Colab **or** local)

The first cell clones + installs the project **when run in Colab**, and is a no-op
locally. The optional second cell installs the `allin1` neural analyzer. The third cell
sets the import path and the knobs.

In [ ]:
# === Google Colab setup — does nothing when run locally ===
# The pipeline + this notebook currently live on the milestone branch (not main),
# so we clone that branch explicitly. After it merges to main, set BRANCH = "main".
REPO_URL = "https://github.com/10cirenehc/CS352-final-SmartAutoDJ.git"
BRANCH   = "milestone-1-pipeline-skeleton"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, subprocess
    name = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
    if not os.path.isdir(name):
        # ffmpeg is already present in Colab; allin1 is NOT installed, so the
        # pipeline's backend="auto" will fall back to librosa (always works).
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL], check=True)
    os.chdir(name)
    subprocess.run(["pip", "install", "-q", "-e", "."], check=True)  # librosa/numpy/scipy/matplotlib/soundfile
    print("Colab: repo ready at", os.getcwd())
else:
    print("Not in Colab — skipping clone/install (use the smartdj conda env).")

### (optional) Better downbeats — `beat_this` (Colab-friendly) or `allin1` (local-only)

`backend="auto"` uses **beat_this** (a CPJKU transformer beat/downbeat tracker) when it is
installed, else falls back to **librosa**. Structure/energy/key now come from our own
`structure` module, so neither neural tracker is required.

* **beat_this** — installs cleanly and runs in Colab on CPU. Recommended upgrade over
  librosa for **real songs**. Install it with the cell below, then set `BACKEND='beat_this'`.
* **allin1** — also gives intro/verse/chorus labels, but its NATTEN/madmom pin matrix is
  **fragile and broken in Colab**; treat it as a **local-only** opt-in (`BACKEND='allin1'`).

On the synthetic click tracks librosa is actually tighter (a metronome is out-of-distribution
for models trained on music), so the upgrade matters mainly for real audio.

In [ ]:
SETUP_BEAT_THIS = False  # flip to True to install beat_this, then set BACKEND='beat_this'

if SETUP_BEAT_THIS:
    import subprocess, sys
    # CPJKU beat_this: accurate transformer downbeats, CPU-friendly, Colab-friendly.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'https://github.com/CPJKU/beat_this/archive/main.zip'], check=True)
    print("beat_this installed — set BACKEND='beat_this' in the knobs cell and re-run.")
else:
    print("Skipping beat_this install — using librosa via backend='auto'.")

# allin1 is local-only (fragile in Colab). To use it locally, install per CLAUDE.md §5
# and set BACKEND='allin1'.

In [ ]:
# Make the repo importable whether run from notebooks/, the repo root, or a Colab clone,
# and whether or not `pip install -e .` has been run.
import os, sys, json
from pathlib import Path

HERE = Path.cwd()
REPO = next((p for p in (HERE, HERE.parent) if (p / "src" / "smartautodj").exists()), HERE)
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)  # so relative paths (data/, outputs/) resolve like the CLI
print("repo:", REPO)

# ----- knobs: edit these, then re-run the cells below -----
SONG_A   = "data/demo_a.wav"   # outgoing track (wav/mp3/flac/m4a/ogg all OK)
SONG_B   = "data/demo_b.wav"   # incoming track
OUT_DIR  = "outputs"
BARS     = 8                   # overlap length in bars (tier 2/3)
BACKEND  = "auto"              # "auto" -> beat_this -> librosa | "beat_this" | "librosa" | "allin1"
CUE      = "energy"            # incoming cue: "energy" (Phase 1) | "novelty" (Phase 2)
BRIDGE   = "riser"             # tier-3 procedural fallback: "riser" | "drum_fill"
BRIDGE_CLIP = None             # path to an AI bridge clip (see generative_musicgen.ipynb); None -> auto/fallback
# The tier-comparison below pins STYLE="default" so tiers 1/2/3 differ only by tier
# (the canonical apples-to-apples objective-metric story). The genre-driven style is
# showcased separately in section 7 ("auto" detects genre and picks the mechanics).
STYLE    = "default"           # "default" | "auto" | dance | urban | band | smooth | ambient
SEED     = 0

In [ ]:
import numpy as np
from IPython.display import Audio, Image, display

from smartautodj import pipeline, DEFAULT_SR
from smartautodj import io as io_mod

print("smartautodj loaded from:", pipeline.__file__)

### Demo clips

`data/` is **gitignored** (no audio in the repo / a fresh Colab clone), so if the demo
clips are missing we synthesize two short 4/4 click tracks at 120 and 124 BPM — the
same deterministic approach the tests use (`tests/conftest.py`). 124/120 = 1.033 sits
inside the default tempo tolerance, so the tier-2 time-stretch actually engages.

To use **your own songs**, drop files into `data/` — `wav`, `mp3`, `flac`, `m4a`, `ogg`
all load fine (decoded via libsndfile / ffmpeg, downmixed to mono and resampled; outputs
are always WAV). In Colab, upload with the file browser or
`from google.colab import files; files.upload()`. Then point `SONG_A`/`SONG_B` at them in
section 0 and re-run.

In [ ]:
def _click_track(bpm, duration=16.0, sr=DEFAULT_SR, seed=0):
    """Metronome-like 4/4 click track with accented downbeats (cf. tests/conftest.py)."""
    rng = np.random.default_rng(seed)
    n = int(duration * sr)
    y = 0.01 * np.sin(2 * np.pi * 110 * np.arange(n) / sr).astype(np.float32)
    beat = 60.0 / bpm
    blen = int(0.04 * sr)
    env = np.exp(-np.linspace(0, 10, blen)).astype(np.float32)
    for k in range(int(duration / beat)):
        i = int(k * beat * sr)
        gain = 1.0 if k % 4 == 0 else 0.6  # accent downbeats
        burst = gain * rng.standard_normal(blen).astype(np.float32) * env
        end = min(i + blen, n)
        y[i:end] += burst[: end - i]
    return (y / (np.max(np.abs(y)) or 1.0) * 0.9).astype(np.float32)

Path("data").mkdir(exist_ok=True)
if not (Path(SONG_A).exists() and Path(SONG_B).exists()):
    io_mod.save_wav(SONG_A, _click_track(120.0, seed=1), DEFAULT_SR)
    io_mod.save_wav(SONG_B, _click_track(124.0, seed=2), DEFAULT_SR)
    print("synthesized demo clips ->", SONG_A, "|", SONG_B)
else:
    print("using existing clips ->", SONG_A, "|", SONG_B)

## 1. Listen to the two input tracks

In [ ]:
y_a, sr = io_mod.load_audio(SONG_A)
y_b, _  = io_mod.load_audio(SONG_B)
print(f"A: {len(y_a)/sr:.1f}s   B: {len(y_b)/sr:.1f}s   sr={sr}")
print("Track A (outgoing):"); display(Audio(y_a, rate=sr))
print("Track B (incoming):"); display(Audio(y_b, rate=sr))

## 2. Run all three tiers

Each `run(...)` writes a WAV, a JSON sidecar, and plots into `outputs/`, and returns a
summary dict (`wav`, `sidecar`, `plots`, `metrics`, `tier`).

In [ ]:
results = {}
for tier in (1, 2, 3):
    results[tier] = pipeline.run(
        song_a=SONG_A, song_b=SONG_B, tier=tier, out_dir=OUT_DIR,
        bars=BARS, bridge_kind=BRIDGE, bridge_clip=BRIDGE_CLIP,
        backend=BACKEND, cue_method=CUE, style=STYLE, seed=SEED,
    )
    print(f"tier {tier}: {results[tier]['wav']}")

## 3. Listen to each transition

In [ ]:
labels = {1: "Tier 1 — baseline crossfade",
          2: "Tier 2 — beat-aligned",
          3: f"Tier 3 — beat-aligned + {BRIDGE} bridge"}
for tier in (1, 2, 3):
    y, _ = io_mod.load_audio(results[tier]["wav"])
    print(labels[tier]); display(Audio(y, rate=sr))

## 4. Compare the objective metrics

Lower is better for all three. Watch tier 1 → tier 2: beat-alignment error and the
post-stretch BPM gap should drop sharply.

In [ ]:
rows = []
for tier in (1, 2, 3):
    m = results[tier]["metrics"]
    rows.append({
        "tier": tier,
        "beat_err_ms":    round(m["beat_alignment"]["mean_abs_error_sec"] * 1000, 2),
        "max_beat_err_ms": round(m["beat_alignment"]["max_abs_error_sec"] * 1000, 2),
        "bpm_gap_before": m["tempo_match"]["bpm_gap_before"],
        "bpm_gap_after":  m["tempo_match"]["bpm_gap_after"],
        "max_db_jump":    m["loudness_continuity"]["max_db_jump"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index("tier"))
except ImportError:
    # pandas isn't a core dep; fall back to a plain print
    cols = ["tier", "beat_err_ms", "max_beat_err_ms", "bpm_gap_before", "bpm_gap_after", "max_db_jump"]
    print("  ".join(cols))
    for r in rows:
        print("  ".join(f"{r[c]:>14}" for c in cols))

## 5. Look at the plots

Waveforms (A tail / B head / output), fade & EQ curves, the chosen transition region
with beat/downbeat markers, and the spectrogram of the output.

In [ ]:
TIER_TO_SHOW = 2  # change to 1 or 3
for p in results[TIER_TO_SHOW]["plots"]:
    print(os.path.basename(p)); display(Image(filename=p))

## 6. Inspect the JSON sidecar

Everything the pipeline decided — analysis (BPM, beats, downbeats, sections), the
transition plan (region, anchors, stretch ratio, EQ), and metrics — is written next to
every WAV so it's inspectable and reusable by the evaluation code.

In [ ]:
side = json.load(open(results[2]["sidecar"]))
print("keys:", list(side.keys()))
print("\nbackend:", side["analysis_a"]["backend"],
      "| A bpm:", side["analysis_a"]["bpm"],
      "| B bpm:", side["analysis_b"]["bpm"])
print("\nplan:")
print(json.dumps(side["plan"], indent=2))

## 8. Scratch

Free space to try your own clips: drop files into `data/`, set `SONG_A`/`SONG_B`
in section 0, and re-run from section 2. Or call `pipeline.run(...)` directly here with
different `bars`, `backend`, `style`, or `bridge_kind`.

In [ ]:
# Genre-driven style: classify both tracks; the detected genre picks the mechanics
# (overlap length, stretch on/off, bass-swap, riser). Uses the optional `transformers`
# model — falls back to the default style if it isn't installed.
g_auto = pipeline.run(
    song_a=SONG_A, song_b=SONG_B, tier=3, out_dir=OUT_DIR,
    bridge_kind=BRIDGE, bridge_clip=BRIDGE_CLIP, backend=BACKEND,
    style="auto", seed=SEED,   # bars/tempo_tol/cue now come from the genre preset
)
print(f"detected genres:  A={g_auto['genre_a']}   B={g_auto['genre_b']}   ->   style: {g_auto['style']}")
gp = json.load(open(g_auto["sidecar"]))["plan"]
print(f"overlap={gp['overlap_sec']:.1f}s   stretch={gp['stretch_ratio']:.3f}   "
      f"eq={'on' if gp['eq_params'] else 'off'}   riser={'yes' if gp['bridge'] else 'no'}")
print("\nGenre-chosen transition:")
display(Audio(io_mod.load_audio(g_auto["wav"])[0], rate=sr))

## 7. Scratch

Free space to try your own clips: drop files into `data/`, set `SONG_A`/`SONG_B`
in section 0, and re-run from section 2. Or call `pipeline.run(...)` directly here with
different `bars`, `backend`, or `bridge_kind`.

In [ ]:
# e.g. a longer, librosa-only tier-2 transition:
# r = pipeline.run(song_a=SONG_A, song_b=SONG_B, tier=2, bars=16, backend="librosa")
# display(Audio(io_mod.load_audio(r["wav"])[0], rate=sr))